# Task 5 — ML-Based Sentiment Analysis (3-Class)

### Introduction
Unlike Lexicon-based sentiment analysis, **ML-based sentiment analysis** trains a classifier on a labeled text corpus.

By training on a dataset of labeled text (Positive, Negative, and Neutral), a model like **Logistic Regression** learns parameters (coefficients) for every unique word in the vocabulary, determining its statistical association with the sentiment class outputs.

### Core Use Cases
- **Domain-Specific Feedback**: Scoring specialized datasets (like medical or product reviews) where generic lexicons fail.
- **High-Accuracy Classification**: Leveraging corpus statistics to capture complex, domain-specific sentiment terms.
- **Feature Importance Diagnostics**: Extracting learned weights to understand key client vocabulary items.

### Step 1 — Build 3-Class Dataset & Train TF-IDF + Logistic Regression Classifier
To build a high-performing 3-class classifier, we construct a balanced dataset by extracting individual sentences from the `movie_reviews` corpus and labelling them (Positive, Negative, Neutral) using VADER scores. This ensures the model learns rich, generic language markers instead of domain-specific noise.

In [1]:
import nltk
from nltk.corpus import movie_reviews
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import random

# Download required datasets
nltk.download('movie_reviews', quiet=True)
nltk.download('vader_lexicon', quiet=True)

# 1. Extract sentences from the movie reviews corpus
sia = SentimentIntensityAnalyzer()
sentences = []
for fileid in movie_reviews.fileids():
    for sent in movie_reviews.sents(fileid):
        sentences.append(' '.join(sent))

# Shuffle sentences for random sampling
random.seed(42)
random.shuffle(sentences)

pos_sents = []
neg_sents = []
neu_sents = []

# 2. Label sentences using VADER polarity scores to form a balanced dataset
for s in sentences:
    if len(s.split()) < 5:  # skip too short sentences
        continue
    score = sia.polarity_scores(s)['compound']
    if score >= 0.35 and len(pos_sents) < 3000:
        pos_sents.append((s, 'positive'))
    elif score <= -0.35 and len(neg_sents) < 3000:
        neg_sents.append((s, 'negative'))
    elif -0.05 <= score <= 0.05 and len(neu_sents) < 3000:
        neu_sents.append((s, 'neutral'))
        
    if len(pos_sents) == 3000 and len(neg_sents) == 3000 and len(neu_sents) == 3000:
        break

# Create DataFrame
df = pd.DataFrame(pos_sents + neg_sents + neu_sents, columns=['text', 'sentiment'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Class Distribution:")
print(df['sentiment'].value_counts())
print("\n")

# Vectorize using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['text'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Logistic Regression (Multi-class)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("ML Classification Report:")
print(classification_report(y_test, y_pred))


Class Distribution:
sentiment
neutral     3000
positive    3000
negative    3000
Name: count, dtype: int64




ML Classification Report:
              precision    recall  f1-score   support

    negative       0.80      0.68      0.73       617
     neutral       0.63      0.82      0.71       572
    positive       0.84      0.74      0.79       611

    accuracy                           0.74      1800
   macro avg       0.76      0.75      0.74      1800
weighted avg       0.76      0.74      0.75      1800



### Step 2 — Extract Sentiment Coefficients
Because this is a multi-class Logistic Regression, it learns a separate set of coefficients for each class.

In [2]:
feature_names = np.array(vectorizer.get_feature_names_out())

for i, class_label in enumerate(model.classes_):
    coefs = model.coef_[i]
    top_indices = np.argsort(coefs)[::-1][:10]
    
    print(f"Top 10 Words for {class_label.upper()}:")
    for idx in top_indices:
        print(f"  {feature_names[idx]:<15} : {coefs[idx]:.4f}")
    print()


Top 10 Words for NEGATIVE:
  bad             : 4.3644
  dead            : 3.4737
  death           : 3.3813
  worst           : 3.2442
  evil            : 3.2365
  war             : 3.1971
  problem         : 3.0317
  wrong           : 2.8218
  horror          : 2.8006
  killer          : 2.7215

Top 10 Words for NEUTRAL:
  dr              : 1.1255
  catch           : 1.1211
  runs            : 1.0758
  mr              : 1.0603
  alien           : 0.8931
  language        : 0.8860
  tell            : 0.8812
  century         : 0.8627
  counting        : 0.8176
  right           : 0.8095

Top 10 Words for POSITIVE:
  like            : 4.9803
  best            : 4.4315
  good            : 4.0301
  love            : 3.8761
  better          : 3.6955
  great           : 3.6330
  funny           : 2.9097
  interesting     : 2.8774
  comedy          : 2.5028
  fun             : 2.4645



### Step 3 — User-Defined Interactive Evaluation Function
Enter your own text below to classify it using the trained 3-class Logistic Regression model.

In [3]:
def evaluate_sentiment_ml(text):
    """
    User-defined evaluation function. Vectorizes custom text and runs Logistic Regression.
    """
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]
    
    print(f"Text Input: '{text[:120]}...'")
    print(f"Prediction: {prediction.upper()}")
    
    probs_str = " | ".join([f"{cls.capitalize()}: {prob*100:.2f}%" for cls, prob in zip(model.classes_, probabilities)])
    print(f"Confidence probabilities -> {probs_str}\n")

# Test custom inputs
evaluate_sentiment_ml("The screening of the film begins tomorrow at 7:30 PM.")
evaluate_sentiment_ml("A masterpiece of cinema. Highly recommend it to anyone who likes good storytelling.")
evaluate_sentiment_ml("The acting was terrible and the plot made absolutely no sense.")


Text Input: 'The screening of the film begins tomorrow at 7:30 PM....'
Prediction: NEUTRAL
Confidence probabilities -> Negative: 17.30% | Neutral: 64.69% | Positive: 18.00%

Text Input: 'A masterpiece of cinema. Highly recommend it to anyone who likes good storytelling....'
Prediction: POSITIVE
Confidence probabilities -> Negative: 10.40% | Neutral: 4.93% | Positive: 84.68%

Text Input: 'The acting was terrible and the plot made absolutely no sense....'
Prediction: NEGATIVE
Confidence probabilities -> Negative: 59.12% | Neutral: 32.37% | Positive: 8.51%

